# Data Collection

We are using a dataset from kaggle for this project

https://www.kaggle.com/datasets/rupakroy/online-payments-fraud-detection-dataset

# Visualising and analysing data

### Importing libaries

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.svm import SVC
import xgboost as xgb
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report, confusion_matrix
import warnings
import pickle

### Read dataset

In [15]:
df = pd.read_csv("fraud detection dataset.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'fraud detection dataset.csv'

In [16]:
df

NameError: name 'df' is not defined

In [17]:
df.drop(["isFlaggedFraud"], axis=1, inplace=True)

NameError: name 'df' is not defined

Here the "isFlagged" is removed as it has only 1 value i.e, 0 and is unecessary

In [18]:
df.head()

NameError: name 'df' is not defined

In [ ]:
df.tail()

In [ ]:
df.columns

We will modify the "matplotlib" plots to "ggplot" style as used in R programming. Additionally, we will also suppress any warning messages generated by matplotlib or any other libraries.

In [ ]:
plt.style.use("ggplot")
warnings.filterwarnings("ignore")

In [ ]:
df["isFraud"].value_counts()

0 means legal transaction and 1 means fraudulent transaction

It is clearly seen that the number of legal transactions is way more than the number of fraudulent transactions which makes it an imbalanced dataset

Hence we need to balance it

In [ ]:
legit = df[df["isFraud"]==0]

In [ ]:
fraud = df[df["isFraud"]==1]

We shall make the number of legal transactions equal to the number of fraudulent transactions to make it even

In [ ]:
legit = legit.sample(n=8213)

In [ ]:
legit.shape, fraud.shape

Now they are equal

In [ ]:
new_df = pd.concat([legit, fraud], axis=0)

In [ ]:
new_df

In [ ]:
new_df.head()

In [ ]:
new_df.tail()

In [ ]:
new_df["isFraud"].value_counts()

In [ ]:
new_df.to_csv('balanced_dataset.csv', index=False, encoding='utf-8')

In [ ]:
corr = df.corr(numeric_only=True)

In [ ]:
corr

In [ ]:
sns.heatmap(corr, annot=True)

### Univariate analysis

The process of understanding data with a single feature is called univariate analysis

In [ ]:
sns.histplot(data=new_df, x="step")

In [ ]:
sns.boxplot(data=new_df, x="step")

In [ ]:
sns.countplot(data=new_df, x="type")

In [ ]:
sns.histplot(data=new_df, x="amount")

In [ ]:
sns.boxplot(data=new_df, x="amount")

In [ ]:
sns.histplot(data=new_df, x="oldbalanceOrg")

In [ ]:
df["nameDest"].value_counts()

In [ ]:
sns.boxplot(data=new_df, x="oldbalanceDest")

In [ ]:
sns.boxplot(data=new_df, x="newbalanceDest")

In [ ]:
sns.countplot(data=new_df, x="isFraud")

### Bivariate analysis

The process of finding the relation between two features is called bivariate analysis

In [ ]:
sns.jointplot(data=new_df, x="newbalanceDest", y="isFraud")

0 means legal transaction and 1 means fraudulent transaction

In [ ]:
sns.countplot(data=new_df, x="type", hue="isFraud")

In [ ]:
sns.boxplot(data=new_df, x="isFraud", y="step")

In [ ]:
sns.boxplot(data=new_df, x="isFraud", y="amount")

In [ ]:
sns.boxplot(data=new_df, x="isFraud", y="oldbalanceOrg")

In [ ]:
sns.boxplot(data=new_df, x="isFraud", y="newbalanceOrig")

In [ ]:
sns.violinplot(data=new_df, x="isFraud", y="oldbalanceDest")

In [ ]:
sns.violinplot(data=new_df, x="isFraud", y="newbalanceDest")

### Descriptive analysis

The process of studying the basic features of data with the statistical process is called descriptive analysis

In [ ]:
new_df.describe(include="all")

# Data preprocessing

In [ ]:
new_df.shape

In [ ]:
new_df.drop(["nameOrig", "nameDest"], axis=1, inplace=True)
df.columns

In [ ]:
new_df.head()

In [ ]:
new_df.tail()

### Checking null values

In [ ]:
new_df.isnull().any()

In [ ]:
new_df.isnull().sum()

Clearly there are no null values

In [ ]:
new_df.info()

### Handling outliers

In [ ]:
sns.boxplot(x=new_df["amount"])

There are outliers in the "amount" column which need to be removed

In [ ]:
q1 = new_df['amount'].quantile(0.25)
q3 = new_df['amount'].quantile(0.75)

iqr = q3 - q1

upper_limit = q3 + 1.5 * iqr
lower_limit = q1 - 1.5 * iqr

mean = new_df['amount'].mean()

# Replace outliers with the mean value
new_df['amount'] = np.where(new_df['amount'] > upper_limit, mean, new_df['amount'])
new_df['amount'] = np.where(new_df['amount'] < lower_limit, mean, new_df['amount'])


In [ ]:
sns.boxplot(x=new_df["amount"])

Now outliers are removed in "amount" columns

In [ ]:
sns.boxplot(x=new_df["oldbalanceOrg"])

The column "oldbalanceOrg" has outliers which need to be treated

In [ ]:
q1 = new_df['oldbalanceOrg'].quantile(0.25)
q3 = new_df['oldbalanceOrg'].quantile(0.75)

iqr = q3 - q1

upper_limit = q3 + 1.5 * iqr
lower_limit = q1 - 1.5 * iqr

mean = new_df['oldbalanceOrg'].mean()

# Replace outliers with the mean value
new_df['oldbalanceOrg'] = np.where(new_df['oldbalanceOrg'] > upper_limit, mean, new_df['oldbalanceOrg'])
new_df['oldbalanceOrg'] = np.where(new_df['oldbalanceOrg'] < lower_limit, mean, new_df['oldbalanceOrg'])

In [ ]:
sns.boxplot(x=new_df["oldbalanceOrg"])

Now "oldbalanceOrg" is free of outliers

In [ ]:
sns.boxplot(x=new_df["newbalanceOrig"])

The column "newbalanceOrig" has outliers which need to be treated

In [ ]:
q1 = new_df['newbalanceOrig'].quantile(0.25)
q3 = new_df['newbalanceOrig'].quantile(0.75)

iqr = q3 - q1

upper_limit = q3 + 1.5 * iqr
lower_limit = q1 - 1.5 * iqr

mean = new_df['newbalanceOrig'].mean()

# Replace outliers with the mean value
new_df['newbalanceOrig'] = np.where(new_df['newbalanceOrig'] > upper_limit, mean, new_df['newbalanceOrig'])
new_df['newbalanceOrig'] = np.where(new_df['newbalanceOrig'] < lower_limit, mean, new_df['newbalanceOrig'])

In [ ]:
sns.boxplot(x=new_df["newbalanceOrig"])

Now "newbalanceOrig" is free of outliers

### Handling categorial or object data using label encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
le = LabelEncoder() 

new_df["type"] = le.fit_transform(new_df["type"])

In [ ]:
new_df["type"].value_counts()

In [ ]:
le.classes_

In [ ]:
map = dict(zip(le.classes_, range(len(le.classes_))))

In [ ]:
map

### Dividing dataset into dependent and independent variable

In [ ]:
x = new_df.drop("isFraud", axis=1) 
y = new_df["isFraud"]

In [ ]:
x.head()

In [ ]:
y.head()

In [ ]:
type(x)

In [ ]:
type(y)

### Splitting data into training and testing set

In [ ]:
from sklearn.model_selection import train_test_split 

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=0, test_size=0.2)

In [ ]:
print(x_train.shape) 
print(x_test.shape) 
print(y_test.shape)
print(y_train.shape)

# Model building

Now that our data is clean, we can train it on different models and pick the best performing model

## 1. Random forest classifier

A Random Forest is an ensemble of decision trees. It builds multiple trees from random subsets of data and features, combines their predictions, and reduces overfitting to create a robust and accurate classification model.

### Import model building libraries

In [ ]:
from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import accuracy_score 

### Initialising the model

In [ ]:
rfc = RandomForestClassifier() 
rfc

### Training and testing the model

In [ ]:
rfc.fit(x_train, y_train)

In [ ]:
# testing accuracy

y_test_predict1 = rfc.predict(x_test) 
test_accuracy = accuracy_score(y_test, y_test_predict1) 
test_accuracy

In [ ]:
# training accuracy 

y_train_predict1 = rfc.predict(x_train) 
train_accuracy = accuracy_score(y_train, y_train_predict1) 
train_accuracy

### Evaluating performance of the model

In [ ]:
pd.crosstab(y_test, y_test_predict1)

In [ ]:
print(classification_report(y_test, y_test_predict1))

## 2. Decision trees

A Decision Tree is a tree-like model that makes decisions by recursively splitting the data based on features, aiming to create homogeneous groups. It's a simple yet interpretable way to perform classification and regression tasks.

### Import model building libraries

In [ ]:
from sklearn.tree import DecisionTreeClassifier 

### Initialising the model

In [ ]:
dtc = DecisionTreeClassifier()
dtc

### Training and testing the model

In [ ]:
dtc.fit(x_train, y_train)

In [11]:
# testing accuracy

y_test_predict2 = dtc.predict(x_test)
test_accuracy = accuracy_score(y_test, y_test_predict2)
test_accuracy

NameError: name 'dtc' is not defined

In [ ]:
# training accuracy

y_train_predict2 = dtc.predict(x_train) 
train_accuracy = accuracy_score(y_train, y_train_predict2) 
train_accuracy

### Evaluating the performance of the model

In [ ]:
pd.crosstab(y_test, y_test_predict2)

In [ ]:
print(classification_report(y_test, y_test_predict2))

## 3. ExtraTrees classifier

An ExtraTrees Classifier (Extremely Randomized Trees Classifier) is an ensemble machine learning model that builds multiple decision trees with a key difference from Random Forest. It adds randomness by selecting random subsets of features and choosing the best split points, which makes it computationally efficient. It combines the predictions from these trees to make accurate classifications and reduce overfitting.

### Import model building libraries

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier

### Initialising the model

In [ ]:
etc = ExtraTreesClassifier() 
etc

### Training and testing the model

In [ ]:
etc.fit(x_train,y_train)

In [ ]:
# testing accuracy

y_test_predict3 = etc.predict(x_test) 
test_accuracy = accuracy_score(y_test, y_test_predict3) 
test_accuracy

In [ ]:
# training accuracy

y_train_predict3 = etc.predict(x_train) 
train_accuracy = accuracy_score(y_train, y_train_predict3) 
train_accuracy

### Evaluating the performance of the model

In [ ]:
pd.crosstab(y_test,y_test_predict3)

In [ ]:
print(classification_report (y_test,y_test_predict3))

## 4. Support vector machine classifier

A Support Vector Machine (SVM) Classifier is a powerful machine learning model used for both classification and regression tasks. It works by finding the best hyperplane that separates different classes in the data, aiming to maximize the margin between the classes. SVMs can handle linear and nonlinear data, making them effective for a wide range of applications, from image recognition to text classification.

### Import model building libraries

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

### Initialising the model

In [ ]:
svc = SVC()
svc

### Training and testing the model

In [ ]:
svc.fit(x_train,y_train)

In [ ]:
# testing accuracy

y_test_predict4 = svc.predict(x_test)
test_accuracy = accuracy_score(y_test, y_test_predict4)
test_accuracy

In [ ]:
# training accuracy

y_train_predict4 = svc.predict(x_train) 
train_accuracy = accuracy_score(y_train, y_train_predict4) 
train_accuracy

### Evaluating the performance of the model

In [ ]:
pd.crosstab(y_test,y_test_predict4)



In [ ]:
from sklearn.metrics import classification_report, confusion_matrix 

print(classification_report(y_test, y_test_predict4))

## 5. xgboost classifier

XGBoost (Extreme Gradient Boosting) Classifier is a popular and powerful machine learning algorithm known for its accuracy and speed. It belongs to the gradient boosting family and combines the predictions of multiple weak learners (typically decision trees) in an iterative manner. XGBoost is designed to minimize prediction errors and can handle complex datasets, making it a top choice for various classification tasks.

### Import model building libraries

In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score

### Initialising the model

In [ ]:
xgb1 = xgb.XGBClassifier()
xgb1

### Training and testing the model

In [ ]:
xgb1.fit(x_train,y_train)

In [13]:
# testing accuracy

y_test_predict5 = xgb1.predict(x_test)
test_accuracy = accuracy_score(y_test, y_test_predict5)
test_accuracy

NameError: name 'xgb1' is not defined

In [ ]:
# training accuracy

y_train_predict5 = svc.predict(x_train) 
train_accuracy = accuracy_score(y_train, y_train_predict5) 
train_accuracy

### Evaluating the performance of the model

In [ ]:
pd.crosstab(y_test, y_test_predict5)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix 

print(classification_report (y_test, y_test_predict5))

## Comparing models

In [12]:
def compareModel():
    print("Train accuracy for RFC: ", accuracy_score(y_train_predict1,y_train)*100) 
    print("Test accuracy for RFC: ", accuracy_score(y_test_predict1,y_test)*100) 
    print("\n")
    print("Train accuracy for DTC: ", accuracy_score(y_train_predict2,y_train)*100) 
    print("Test accuracy for DTC: ", accuracy_score(y_test_predict2,y_test)*100) 
    print("\n")
    print("Train accuracy for ETC: ", accuracy_score(y_train_predict3,y_train)*100) 
    print("Test accuracy for ETC: ", accuracy_score (y_test_predict3,y_test)*100) 
    print("\n")
    print("Train accuracy for SVC: ", accuracy_score(y_train_predict4,y_train)*100) 
    print("Test accuracy for SVC: ", accuracy_score(y_test_predict4,y_test)*100) 
    print("\n")
    print("Train accuracy for XGB: ", accuracy_score(y_train_predict5,y_train)*100) 
    print("Test accuracy for XGB: ", accuracy_score(y_test_predict5,y_test)*100)


In [ ]:
compareModel()

On comparing the training and testing of 5 different models trained by 5 different algorithms, we have found that XGBoost Classifier is the best model as it has the highest testing accuracy and is not overfitting. 

## Save the model

In [ ]:
import pickle 
pickle.dump(xgb1, open('payments.pkl', 'wb'))